In [1]:
!git clone -q https://github.com/pdhung3012/T2J-bench-v2.git /content/T2J-bench-v2

from pathlib import Path

repo = Path("/content/T2J-bench-v2")

for folder in ["datasets", "jax_qwen_instruct"]:
    print(f"\n--- {folder}: RMS-related files ---")
    for path in sorted((repo / folder).rglob("*")):
        if path.is_file() and "rms" in path.name.lower():
            print(path.relative_to(repo))


--- datasets: RMS-related files ---
datasets/kernelbench/level1/36_RMSNorm_.py
datasets/kernelbench/level1/__pycache__/36_RMSNorm_.cpython-311.pyc
datasets/torchleet-llms/RMS-Norm.py
datasets/torchleet-llms/__pycache__/RMS-Norm.cpython-311.pyc

--- jax_qwen_instruct: RMS-related files ---
jax_qwen_instruct/kernelbench/level1/36_RMSNorm_.py
jax_qwen_instruct/kernelbench/level1/__pycache__/36_RMSNorm_.cpython-311.pyc
jax_qwen_instruct/kernelbench/level1/__pycache__/36_RMSNorm_.cpython-312.pyc
jax_qwen_instruct/torchleet-llms/RMS-Norm.py
jax_qwen_instruct/torchleet-llms/__pycache__/RMS-Norm.cpython-311.pyc
jax_qwen_instruct/torchleet-llms/__pycache__/RMS-Norm.cpython-312.pyc


In [2]:
for folder in ["datasets", "jax_qwen_instruct"]:
    for path in sorted((repo / folder).rglob("*")):
        if path.is_file() and "rms" in path.name.lower():
            if path.suffix in [".py", ".txt", ".json"]:
                print(f"\n{'=' * 60}")
                print(path.relative_to(repo))
                print("=" * 60)
                print(path.read_text())


datasets/kernelbench/level1/36_RMSNorm_.py
import torch
import torch.nn as nn

class Model(nn.Module):
    """
    Simple model that performs RMS Normalization.
    """
    def __init__(self, num_features: int, eps: float = 1e-5):
        """
        Initializes the RMSNorm layer.

        Args:
            num_features (int): Number of features in the input tensor.
            eps (float, optional): A small value added to the denominator to avoid division by zero. Defaults to 1e-5.
        """
        super(Model, self).__init__()
        self.num_features = num_features
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Applies RMS Normalization to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, num_features, *).

        Returns:
            torch.Tensor: Output tensor with RMS Normalization applied, same shape as input.
        """
        # Calculate the RMS along the feature dimens

In [3]:
from pathlib import Path
import json
import subprocess

# Download the repository if it is not already there.
repo = Path("/content/T2J-bench-v2")

if not repo.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/pdhung3012/T2J-bench-v2.git",
            str(repo),
        ],
        check=True,
    )

# Create a folder for our demo notes.
demo_dir = Path("/content/rms_repair_demo")
demo_dir.mkdir(exist_ok=True)

# Record the exact repository version used.
commit = subprocess.check_output(
    ["git", "-C", str(repo), "rev-parse", "HEAD"],
    text=True,
).strip()

print("Repository version:", commit)
(demo_dir / "repository_version.txt").write_text(commit + "\n")

# Find RMS-related files in both datasets.
sections = []

for folder in ["datasets", "jax_qwen_instruct"]:
    matches = [
        path
        for path in sorted((repo / folder).rglob("*"))
        if path.is_file()
        and "rms" in str(path.relative_to(repo / folder)).lower()
        and path.suffix.lower() in {".py", ".ipynb", ".txt", ".json"}
    ]

    print(f"\n{folder}: {len(matches)} matching files")

    for path in matches:
        heading = f"\n{'=' * 70}\n{path.relative_to(repo)}\n{'=' * 70}\n"

        if path.suffix.lower() == ".ipynb":
            notebook = json.loads(path.read_text(encoding="utf-8"))
            parts = []

            for index, cell in enumerate(notebook.get("cells", [])):
                source = cell.get("source", "")
                if isinstance(source, list):
                    source = "".join(source)

                parts.append(
                    f"\n--- Cell {index}: {cell['cell_type']} ---\n{source}\n"
                )

            content = "".join(parts)
        else:
            content = path.read_text(encoding="utf-8")

        section = heading + content
        print(section)
        sections.append(section)

# Save the source inspection for our documentation.
inspection_file = demo_dir / "source_inspection.txt"
inspection_file.write_text(
    "\n".join(sections),
    encoding="utf-8",
)

print("\nSaved source inspection:", inspection_file)

if not sections:
    print("\nNo RMS files found. Available files:")
    for folder in ["datasets", "jax_qwen_instruct"]:
        for path in sorted((repo / folder).rglob("*")):
            if path.is_file():
                print(path.relative_to(repo))

Repository version: ca83a53c267ac475a880409d94973d302313dce4

datasets: 2 matching files

datasets/kernelbench/level1/36_RMSNorm_.py
import torch
import torch.nn as nn

class Model(nn.Module):
    """
    Simple model that performs RMS Normalization.
    """
    def __init__(self, num_features: int, eps: float = 1e-5):
        """
        Initializes the RMSNorm layer.

        Args:
            num_features (int): Number of features in the input tensor.
            eps (float, optional): A small value added to the denominator to avoid division by zero. Defaults to 1e-5.
        """
        super(Model, self).__init__()
        self.num_features = num_features
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Applies RMS Normalization to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, num_features, *).

        Returns:
            torch.Tensor: Output tensor with RMS Normalization appli

In [4]:
from pathlib import Path
import contextlib
import io
import json
import traceback
import numpy as np
import torch
import jax
import jax.numpy as jnp

repo = Path("/content/T2J-bench-v2")
demo_dir = Path("/content/rms_repair_demo")
demo_dir.mkdir(exist_ok=True)

torch_path = repo / "datasets/torchleet-llms/RMS-Norm.py"
jax_path = repo / "jax_qwen_instruct/torchleet-llms/RMS-Norm.py"

torch_source = torch_path.read_text()
jax_source = jax_path.read_text()

# Keep untouched copies of the two source files.
(demo_dir / "pytorch_original.py").write_text(torch_source)
(demo_dir / "jax_generated_original.py").write_text(jax_source)

def run_source(source, filename):
    namespace = {"__name__": "__main__"}
    output = io.StringIO()
    error = None

    with contextlib.redirect_stdout(output), contextlib.redirect_stderr(output):
        try:
            exec(compile(source, filename, "exec"), namespace)
        except Exception:
            error = traceback.format_exc()

    return namespace, {
        "success": error is None,
        "stdout": output.getvalue(),
        "error": error,
    }

torch_namespace, torch_run = run_source(torch_source, str(torch_path))
_, jax_run = run_source(jax_source, str(jax_path))

original_runs = {"pytorch": torch_run, "generated_jax": jax_run}
(demo_dir / "original_runs.json").write_text(
    json.dumps(original_runs, indent=2)
)

for name, result in original_runs.items():
    print(f"\n--- {name} ---")
    print("Execution success:", result["success"])
    print(result["stdout"])
    if result["error"]:
        print(result["error"])

assert torch_run["success"], "PyTorch reference failed; inspect its error first."


--- pytorch ---
Execution success: True
torch.Size([3, 5])


--- generated_jax ---
Execution success: False

Traceback (most recent call last):
  File "/tmp/ipykernel_2460/4003712116.py", line 32, in run_source
    exec(compile(source, filename, "exec"), namespace)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/T2J-bench-v2/jax_qwen_instruct/torchleet-llms/RMS-Norm.py", line 3, in <module>
    from jax.experimental import optimizers
ImportError: cannot import name 'optimizers' from 'jax.experimental' (/usr/local/lib/python3.13/dist-packages/jax/experimental/__init__.py)



In [5]:
import importlib

diagnostics = {}

checks = {
    "optimizers_import": lambda: importlib.import_module(
        "jax.experimental.optimizers"
    ),
    "stax_import_and_names": lambda: exec(
        "from jax.experimental.stax import "
        "Dense, Relu, Normalize, Exp, Add, Constant",
        {},
    ),
    "jnp_randn": lambda: jnp.randn(3, 5),
}

for name, check in checks.items():
    try:
        check()
        diagnostics[name] = {"success": True, "error": None}
    except Exception as error:
        diagnostics[name] = {
            "success": False,
            "error": f"{type(error).__name__}: {error}",
        }

    print(name, ":", diagnostics[name])

(demo_dir / "diagnostics.json").write_text(
    json.dumps(diagnostics, indent=2)
)

optimizers_import : {'success': False, 'error': "ModuleNotFoundError: No module named 'jax.experimental.optimizers'"}
stax_import_and_names : {'success': False, 'error': "ModuleNotFoundError: No module named 'jax.experimental.stax'"}
jnp_randn : {'success': False, 'error': "AttributeError: module 'jax.numpy' has no attribute 'randn'"}


387

In [6]:
fixed_source = '''import jax
import jax.numpy as jnp

class RMSNorm:
    def __init__(self, dim: int, eps: float = 1e-8):
        self.eps = eps
        self.scale = jnp.ones(dim)

    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        norm = jnp.sqrt(
            jnp.mean(x ** 2, axis=-1, keepdims=True) + self.eps
        )
        return (x / norm) * self.scale

x = jax.random.normal(
    jax.random.PRNGKey(0), shape=(3, 5)
)
rmsnorm = RMSNorm(dim=5)
out = rmsnorm(x)
print(out.shape)
assert out.shape == (3, 5), "Output shape mismatch"
'''

fixed_path = demo_dir / "jax_repaired.py"
fixed_path.write_text(fixed_source)

fixed_namespace, fixed_run = run_source(fixed_source, str(fixed_path))

print("Repaired execution success:", fixed_run["success"])
print(fixed_run["stdout"])

if fixed_run["error"]:
    print(fixed_run["error"])

(demo_dir / "repaired_run.json").write_text(
    json.dumps(fixed_run, indent=2)
)

assert fixed_run["success"], "Inspect the repaired-code error before continuing."

Repaired execution success: True
(3, 5)



In [9]:
import numpy as np
import json
import shutil

TorchRMSNorm = torch_namespace["RMSNorm"]
JaxRMSNorm = fixed_namespace["RMSNorm"]

cases = {
    "mixed_values": np.array([
        [1, 2, 3, 4, 5],
        [-2, -1, 0, 1, 2],
        [0.5, -0.5, 1.5, -1.5, 0]
    ], dtype=np.float32),
    "zeros": np.zeros((3, 5), dtype=np.float32),
    "constant_values": np.full((3, 5), 2, dtype=np.float32),
    "small_values": np.full((3, 5), 1e-6, dtype=np.float32),
    "three_dimensions": np.arange(
        -10, 10, dtype=np.float32
    ).reshape(2, 2, 5),
}

results = []

for name, x in cases.items():
    with torch.no_grad():
        expected = TorchRMSNorm(x.shape[-1])(
            torch.from_numpy(x.copy())
        ).numpy()

    actual = np.asarray(JaxRMSNorm(x.shape[-1])(jnp.asarray(x)))
    matches = (
        actual.shape == expected.shape
        and np.allclose(actual, expected, rtol=1e-5, atol=1e-6)
    )
    error = float(np.max(np.abs(actual - expected)))

    results.append({
        "case": name,
        "matches": bool(matches),
        "max_absolute_error": error,
    })

    np.savez(
        demo_dir / f"comparison_{name}.npz",
        input=x,
        pytorch_output=expected,
        repaired_jax_output=actual,
    )

    print(f"{name}: {'PASS' if matches else 'FAIL'} | error={error:.8g}")

# Check the original sum-based formula separately.
x = cases["mixed_values"]

with torch.no_grad():
    expected = TorchRMSNorm(5)(torch.from_numpy(x.copy())).numpy()

wrong = np.asarray(
    jnp.asarray(x) / jnp.sqrt(
        jnp.sum(jnp.asarray(x) ** 2, axis=-1, keepdims=True) + 1e-8
    )
)

sum_matches = bool(np.allclose(wrong, expected, rtol=1e-5, atol=1e-6))
print("\nOriginal sum formula matches PyTorch:", sum_matches)

comparison = {
    "tests": results,
    "original_sum_formula_matches": sum_matches,
    "rtol": 1e-5,
    "atol": 1e-6,
}

(demo_dir / "comparison_results.json").write_text(
    json.dumps(comparison, indent=2)
)

versions = {
    "torch": str(torch.__version__),
    "jax": str(jax.__version__),
    "numpy": str(np.__version__),
}
(demo_dir / "environment.json").write_text(json.dumps(versions, indent=2))

passed = sum(r["matches"] for r in results)
commit = (demo_dir / "repository_version.txt").read_text().strip()

lines = [
    "# RMSNorm JAX repair",
    "",
    f"Repository commit: {commit}",
    "Source pair: datasets/torchleet-llms/RMS-Norm.py and",
    "jax_qwen_instruct/torchleet-llms/RMS-Norm.py",
    "",
    "## Original execution",
    f"PyTorch execution success: {torch_run['success']}",
    f"Generated JAX execution success: {jax_run['success']}",
    "```text",
    jax_run["error"] or "No exception",
    "```",
    "",
    "## Fixes",
    "1. Removed unused optimizers and stax imports.",
    "2. Replaced jnp.randn with jax.random.normal and an explicit key.",
    "3. Replaced sum of squares with mean of squares to match PyTorch.",
    "Independent API errors are recorded in diagnostics.json.",
    "",
    "## Numerical validation",
    f"Repaired execution success: {fixed_run['success']}",
    f"Passed {passed}/{len(results)} shared-input tests.",
    "Tolerance: rtol=1e-5, atol=1e-6.",
    "| Case | Matches | Maximum absolute error |",
    "|---|---|---|",
]

for r in results:
    lines.append(
        f"| {r['case']} | {r['matches']} | {r['max_absolute_error']:.8g} |"
    )

lines.extend([
    "",
    f"Original sum formula matches on mixed inputs: {sum_matches}",
    "Inputs and outputs are saved in comparison_*.npz.",
    "",
    "## Limitations",
    "Validation covers five forward-output cases with scale initialized to ones.",
    "It does not prove correctness for all inputs or validate training,",
    "gradients, or optimizer updates.",
])

(demo_dir / "RMSNorm_fix_log.md").write_text("\n".join(lines))

archive = shutil.make_archive(
    "/content/rms_repair_demo", "zip", root_dir=demo_dir
)

print(f"\nFINAL RESULT: {passed}/{len(results)} tests passed")
print("Saved report and evidence:", archive)

from google.colab import files
files.download(archive)

mixed_values: PASS | error=5.9604645e-08
zeros: PASS | error=0
constant_values: PASS | error=0
small_values: PASS | error=0
three_dimensions: PASS | error=5.9604645e-08

Original sum formula matches PyTorch: False

FINAL RESULT: 5/5 tests passed
Saved report and evidence: /content/rms_repair_demo.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>